In [1]:
from hw2fintools import gurufocus as gf
import os
from dotenv import load_dotenv
import pandas as pd
import re


load_dotenv()
path_stockdata = os.path.join(os.environ.get('judgement_day'), 'Data--StockWatchList')
date_pattern = r"(\d\d\d\d-\d\d-\d\d)--"


In [2]:
ticker = input('Enter ticker symbol: ')
print('Ticker is: ', ticker)

Ticker is:  AMD


In [3]:
path_ticker = os.path.join(path_stockdata, ticker.upper())
div_list = []

for item in os.listdir(path_ticker):
    if 'gf-raw-dividend_history' in item and item.endswith('.csv') and not item.startswith('._'):
        div_list.append(item)

div_list_sorted = sorted(div_list, reverse=True)
div_current = div_list_sorted[0]

div_path = os.path.join(path_ticker, div_current)
div_path

'/Volumes/JudgmentDay/StockData/AMD/2025-12-09--AMD--gf-raw-dividend_history.csv'

In [4]:
price_list = []

for item in os.listdir(path_ticker):
    if 'gf-raw-price_history' in item and item.endswith('.csv') and not item.startswith('._'):
        price_list.append(item)

price_list_sorted = sorted(price_list, reverse=True)
price_current = price_list_sorted[0]

price_path = os.path.join(path_ticker, price_current)
price_path

'/Volumes/JudgmentDay/StockData/AMD/2025-12-09--AMD--gf-raw-price_history.csv'

In [5]:
div_date = re.search(date_pattern, div_path)
price_date = re.search(date_pattern, price_path)

div_match = div_date.group(1)
price_match = price_date.group(1)

if div_match == price_match:
    print('date match')
else:
    print('date mismatch')


date match


In [6]:
div_df0 = gf.div_hist_s1v1(div_path)


In [7]:
div_df1 = div_df0.loc[div_df0['DivType'] == 'regular']
div_df1 = div_df1.drop(columns=['DivRecordDate', 'DivDeclareDate', 'DivPayDate'])
div_df1 = div_df1.rename(columns={'ExDivDate': 'Date'})
div_df1

,DivAmount,Date,DivType,Currency,DivFrequency


In [8]:
price_df0 = gf.price_hist_s2v1(price_path)

In [9]:
merged_df0 = pd.merge(price_df0, div_df1, on='Date', how='left')
merged_df0['DivAmount'] = merged_df0['DivAmount'].fillna(0)
merged_df0['DivFrequency'] = merged_df0['DivFrequency'].fillna(div_df1.iloc[0]['DivFrequency'])
merged_df0['DivType'] = merged_df0['DivType'].fillna(div_df1.iloc[0]['DivType'])
merged_df0['DivPayDeclared'] = merged_df0['DivAmount'].fillna(0)
merged_df0

IndexError: single positional indexer is out-of-bounds

In [10]:
div_var = 0

for index, row in merged_df0.iterrows():
    if row['DivAmount'] > 0:
        div_var = row['DivAmount']

    else:
        merged_df0.at[index, 'DivAmount'] = div_var


merged_df0['FwdDiv'] = merged_df0['DivFrequency'] * merged_df0['DivAmount']
merged_df0['FwdDivYield'] = merged_df0['FwdDiv'] / merged_df0['PricePerShare']

merged_df0

,Date,PricePerShare,DivAmount,DivType,Currency,DivFrequency,DivPayDeclared,FwdDiv,FwdDivYield
0,1986-04-02,0.804700,0.00,regular,NaN,4.0,0.0,0.00,0.000000
1,1986-04-03,0.906300,0.00,regular,NaN,4.0,0.0,0.00,0.000000
2,1986-04-04,0.906300,0.00,regular,NaN,4.0,0.0,0.00,0.000000
3,1986-04-07,0.890699,0.00,regular,NaN,4.0,0.0,0.00,0.000000
4,1986-04-08,0.906300,0.00,regular,NaN,4.0,0.0,0.00,0.000000
...,...,...,...,...,...,...,...,...,...
9995,2025-12-02,102.470000,1.27,regular,NaN,4.0,0.0,5.08,0.049575
9996,2025-12-03,105.050000,1.27,regular,NaN,4.0,0.0,5.08,0.048358
9997,2025-12-04,105.790000,1.27,regular,NaN,4.0,0.0,5.08,0.048020
9998,2025-12-05,106.580000,1.27,regular,NaN,4.0,0.0,5.08,0.047664


In [11]:
aggr_df0 = merged_df0
aggr_df0 = aggr_df0.set_index('Date')
aggr_df1 = aggr_df0.groupby(aggr_df0.index.year).agg(
    SharePriceMin=pd.NamedAgg(column='PricePerShare', aggfunc='min'),
    SharePriceMax=pd.NamedAgg(column='PricePerShare', aggfunc='max'),
    SharePriceMean=pd.NamedAgg(column='PricePerShare', aggfunc='mean'),
    SharePriceMedian=pd.NamedAgg(column='PricePerShare', aggfunc='median'),
    DivYieldMin=pd.NamedAgg(column='FwdDivYield', aggfunc='min'),
    DivYieldMax=pd.NamedAgg(column='FwdDivYield', aggfunc='max'),
    DivYieldMean=pd.NamedAgg(column='FwdDivYield', aggfunc='mean'),
    DivYieldMedian=pd.NamedAgg(column='FwdDivYield', aggfunc='median'),
    DividendPaidTotalCy=pd.NamedAgg('DivPayDeclared', aggfunc='sum')
)
aggr_df1

,SharePriceMin,SharePriceMax,SharePriceMean,SharePriceMedian,DivYieldMin,DivYieldMax,DivYieldMean,DivYieldMedian,DividendPaidTotalCy
Date,,,,,,,,,
1986,0.8047,1.336000,1.030539,1.00790,0.000000,0.013428,0.004010,0.005120,0.00580
1987,0.5547,1.586000,1.116229,1.17970,0.008071,0.028160,0.012601,0.010850,0.01400
1988,0.6641,1.101600,0.938994,0.96100,0.015977,0.026502,0.019203,0.018464,0.01980
1989,1.0860,1.828199,1.451401,1.44540,0.015360,0.024309,0.018648,0.018415,0.02920
1990,0.8985,1.953200,1.521684,1.57820,0.019250,0.041848,0.025580,0.023825,0.03820
1991,1.1719,2.812500,1.852341,1.81250,0.016071,0.034133,0.022305,0.022069,0.04130
1992,2.0625,3.062500,2.498346,2.46490,0.014759,0.021915,0.018330,0.018544,0.04710
1993,2.5469,4.086000,3.219620,3.01570,0.012922,0.020731,0.016793,0.017600,0.05590
1994,3.1563,4.781300,3.808117,3.76180,0.013636,0.021879,0.017308,0.017368,0.06890


In [12]:
merged_df0.to_csv(os.path.join(path_ticker, ticker.upper() + '--DivPrice_History--s3v1.csv'))

In [13]:
aggr_df1.to_csv(os.path.join(path_ticker, ticker.upper() + '--Aggregate_Cy_DivPrice_History--s4v1.csv'))